In [1]:
# ============================================================
# LAB 3: MULTI-SOURCE RETAIL SALES DATA INTEGRATION & ANALYSIS
# Platform: R / Google Colab
# ============================================================

# Install required packages
install.packages(c(
  "tidyverse",
  "jsonlite",
  "readxl",
  "writexl",
  "DBI",
  "RSQLite"
), repos = "https://cloud.r-project.org")

# Load libraries
library(tidyverse)
library(jsonlite)
library(readxl)
library(writexl)
library(DBI)
library(RSQLite)

# Download UCI Online Retail Dataset
url <- "https://archive.ics.uci.edu/static/public/352/online+retail.zip"
zip_file <- "online_retail.zip"

download.file(url, zip_file, mode = "wb")

# Extract dataset
unzip(zip_file)

# Check extracted files
list.files()

# The UCI dataset is normally provided as Online Retail.xlsx
raw_data <- read_excel("Online Retail.xlsx")

cat("Original dataset dimensions:\n")
cat("Rows:", nrow(raw_data), "\n")
cat("Columns:", ncol(raw_data), "\n")

head(raw_data)

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘jsonlite’


The following object is masked from ‘package:purrr’:

    flatten




[1] "Online Retail.xlsx" "online_retail.zip"  "sample_data"

Original dataset dimensions:
Rows: 541909 
Columns: 8 


InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
<chr>,<chr>,<chr>,<dbl>,<dttm>,<dbl>,<dbl>,<chr>
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850,United Kingdom


In [2]:
# ============================================================
# TASK 1: IMPORT AND CLEAN THE DATA
# ============================================================

# ------------------------------------------------------------
# STEP 1: Prepare the three required source files
# ------------------------------------------------------------

# Rename columns to convenient names
retail_raw <- raw_data %>%
  rename(
    InvoiceNo = InvoiceNo,
    StockCode = StockCode,
    Description = Description,
    Quantity = Quantity,
    InvoiceDate = InvoiceDate,
    UnitPrice = UnitPrice,
    CustomerID = CustomerID,
    Country = Country
  )

# Create transactions.csv
transactions <- retail_raw %>%
  select(
    InvoiceNo,
    StockCode,
    CustomerID,
    Quantity,
    InvoiceDate
  )

write_csv(transactions, "transactions.csv")

# Create products.json
products <- retail_raw %>%
  select(StockCode, Description, UnitPrice) %>%
  filter(!is.na(StockCode),
         !is.na(Description),
         !is.na(UnitPrice),
         UnitPrice > 0) %>%
  group_by(StockCode, Description) %>%
  summarise(
    UnitPrice = median(UnitPrice),
    .groups = "drop"
  )

write_json(products, "products.json", pretty = TRUE)

# Create customers.xlsx
customers <- retail_raw %>%
  select(CustomerID, Country) %>%
  filter(!is.na(CustomerID),
         !is.na(Country)) %>%
  distinct()

write_xlsx(customers, "customers.xlsx")


# ------------------------------------------------------------
# STEP 2: IMPORT CSV, JSON AND EXCEL
# ------------------------------------------------------------

transactions_imported <- read_csv(
  "transactions.csv",
  show_col_types = FALSE
)

products_imported <- fromJSON("products.json")

customers_imported <- read_excel("customers.xlsx")


# ------------------------------------------------------------
# STEP 3: INSPECT DATA
# ------------------------------------------------------------

cat("\n================ DATA DIMENSIONS ================\n")

cat("\nTransactions:\n")
print(dim(transactions_imported))

cat("\nProducts:\n")
print(dim(products_imported))

cat("\nCustomers:\n")
print(dim(customers_imported))

cat("\nTransactions structure:\n")
str(transactions_imported)

cat("\nProducts structure:\n")
str(products_imported)

cat("\nCustomers structure:\n")
str(customers_imported)


# ------------------------------------------------------------
# STEP 4: MISSING VALUE CHECK
# ------------------------------------------------------------

cat("\n================ MISSING VALUES ================\n")

cat("\nTransactions missing values:\n")
print(colSums(is.na(transactions_imported)))

cat("\nProducts missing values:\n")
print(colSums(is.na(products_imported)))

cat("\nCustomers missing values:\n")
print(colSums(is.na(customers_imported)))


# ------------------------------------------------------------
# STEP 5: CLEAN TRANSACTIONS
# ------------------------------------------------------------

transactions_clean <- transactions_imported %>%
  filter(
    !is.na(InvoiceNo),
    !is.na(StockCode),
    !is.na(CustomerID),
    !is.na(Quantity),
    !is.na(InvoiceDate)
  ) %>%
  filter(Quantity > 0) %>%
  distinct()


# ------------------------------------------------------------
# STEP 6: CLEAN PRODUCTS
# ------------------------------------------------------------

products_clean <- products_imported %>%
  filter(
    !is.na(StockCode),
    !is.na(Description),
    !is.na(UnitPrice),
    UnitPrice > 0
  ) %>%
  distinct()


# ------------------------------------------------------------
# STEP 7: CLEAN CUSTOMERS
# ------------------------------------------------------------

customers_clean <- customers_imported %>%
  filter(
    !is.na(CustomerID),
    !is.na(Country)
  ) %>%
  distinct()


# ------------------------------------------------------------
# STEP 8: REMOVE DUPLICATE PRODUCT STOCK CODES
# ------------------------------------------------------------

products_clean <- products_clean %>%
  group_by(StockCode) %>%
  summarise(
    Description = first(Description),
    UnitPrice = median(UnitPrice),
    .groups = "drop"
  )


# ------------------------------------------------------------
# STEP 9: ADD UNIT PRICE TO TRANSACTIONS
# ------------------------------------------------------------

transactions_clean <- transactions_clean %>%
  left_join(
    products_clean %>% select(StockCode, UnitPrice),
    by = "StockCode"
  )


# ------------------------------------------------------------
# STEP 10: REMOVE TRANSACTIONS WITH INVALID PRICES
# ------------------------------------------------------------

transactions_clean <- transactions_clean %>%
  filter(
    !is.na(UnitPrice),
    UnitPrice > 0
  )


# ------------------------------------------------------------
# STEP 11: CREATE REVENUE
# ------------------------------------------------------------

transactions_clean <- transactions_clean %>%
  mutate(
    Revenue = Quantity * UnitPrice
  )


# ------------------------------------------------------------
# STEP 12: FINAL CLEANING SUMMARY
# ------------------------------------------------------------

cat("\n================ CLEANING SUMMARY ================\n")

cat("\nOriginal rows:", nrow(transactions_imported))
cat("\nClean transaction rows:", nrow(transactions_clean))
cat("\nRemoved rows:", nrow(transactions_imported) - nrow(transactions_clean))

cat("\n\nDuplicate transaction records removed:",
    nrow(transactions_imported) - nrow(distinct(transactions_imported)))

cat("\n\nInvalid/zero quantities removed:",
    sum(transactions_imported$Quantity <= 0, na.rm = TRUE))

cat("\n\nInvalid/zero prices in product data:",
    sum(products_imported$UnitPrice <= 0, na.rm = TRUE))

cat("\n\nFinal Revenue column created successfully.\n")

cat("\nCleaning decisions:\n")
cat("1. Records with missing essential transaction fields were removed.\n")
cat("2. Duplicate transaction records were removed.\n")
cat("3. Transactions with zero or negative quantities were removed.\n")
cat("4. Products with missing or non-positive prices were removed.\n")
cat("5. Customers without CustomerID or Country were excluded.\n")
cat("6. Revenue was calculated as Quantity × UnitPrice.\n")

cat("\nFinal cleaned transaction sample:\n")
print(head(transactions_clean))


================ DATA DIMENSIONS ================

Transactions:
[1] 541909      5

Products:
[1] 4174    3

Customers:
[1] 4380    2

Transactions structure:
spc_tbl_ [541,909 × 5] (S3: spec_tbl_df/tbl_df/tbl/data.frame)
 $ InvoiceNo  : chr [1:541909] "536365" "536365" "536365" "536365" ...
 $ StockCode  : chr [1:541909] "85123A" "71053" "84406B" "84029G" ...
 $ CustomerID : num [1:541909] 17850 17850 17850 17850 17850 ...
 $ Quantity   : num [1:541909] 6 6 8 6 6 2 6 6 6 32 ...
 $ InvoiceDate: POSIXct[1:541909], format: "2010-12-01 08:26:00" "2010-12-01 08:26:00" ...
 - attr(*, "spec")=
  .. cols(
  ..   InvoiceNo = col_character(),
  ..   StockCode = col_character(),
  ..   CustomerID = col_double(),
  ..   Quantity = col_double(),
  ..   InvoiceDate = col_datetime(format = "")
  .. )
 - attr(*, "problems")=<pointer: 0x589d5eaf8120> 

Products structure:
'data.frame':	4174 obs. of  3 variables:
 $ StockCode  : chr  "10002" "10080" "10120" "10123C" ...
 $ Description: chr  "INFLATABL

In [3]:
# ============================================================
# TASK 2: INTEGRATE MULTIPLE DATA SOURCES
# ============================================================

# ------------------------------------------------------------
# STEP 1: JOIN TRANSACTIONS WITH PRODUCT INFORMATION
# ------------------------------------------------------------

sales_products <- transactions_clean %>%
  left_join(
    products_clean,
    by = "StockCode",
    suffix = c("_transaction", "_product")
  )


# ------------------------------------------------------------
# STEP 2: JOIN WITH CUSTOMER INFORMATION
# ------------------------------------------------------------

final_retail_data <- sales_products %>%
  left_join(
    customers_clean,
    by = "CustomerID"
  )


# ------------------------------------------------------------
# STEP 3: CLEAN COLUMN NAMES
# ------------------------------------------------------------

final_retail_data <- final_retail_data %>%
  select(
    InvoiceNo,
    StockCode,
    CustomerID,
    Quantity,
    InvoiceDate,
    Description,
    UnitPrice = UnitPrice_transaction,
    Country,
    Revenue
  )


# ------------------------------------------------------------
# STEP 4: VERIFY FINAL DIMENSIONS
# ------------------------------------------------------------

cat("\n================ FINAL DATASET DIMENSIONS ================\n")

cat("Rows:", nrow(final_retail_data), "\n")
cat("Columns:", ncol(final_retail_data), "\n")

cat("\nColumn names:\n")
print(names(final_retail_data))


# ------------------------------------------------------------
# STEP 5: IDENTIFY UNMATCHED PRODUCT RECORDS
# ------------------------------------------------------------

unmatched_products <- transactions_clean %>%
  anti_join(
    products_clean,
    by = "StockCode"
  )

cat("\n================ UNMATCHED PRODUCTS ================\n")
cat("Number of unmatched product records:",
    nrow(unmatched_products), "\n")


# ------------------------------------------------------------
# STEP 6: IDENTIFY UNMATCHED CUSTOMER RECORDS
# ------------------------------------------------------------

unmatched_customers <- transactions_clean %>%
  anti_join(
    customers_clean,
    by = "CustomerID"
  )

cat("\n================ UNMATCHED CUSTOMERS ================\n")
cat("Number of unmatched customer records:",
    nrow(unmatched_customers), "\n")


# ------------------------------------------------------------
# STEP 7: VERIFY MISSING VALUES AFTER INTEGRATION
# ------------------------------------------------------------

cat("\n================ INTEGRATED DATA CHECK ================\n")

print(colSums(is.na(final_retail_data)))


# ------------------------------------------------------------
# STEP 8: DISPLAY FINAL DATASET
# ------------------------------------------------------------

cat("\n================ FINAL INTEGRATED DATA ================\n")

print(head(final_retail_data, 10))


# ------------------------------------------------------------
# STEP 9: JOIN JUSTIFICATION
# ------------------------------------------------------------

cat("\n================ JOIN JUSTIFICATION ================\n")

cat(
  "A LEFT JOIN was selected because all valid transaction records\n",
  "must be retained while product and customer information are added.\n",
  "This prevents valid sales transactions from being lost during integration.\n"
)

Warning message in left_join(., customers_clean, by = "CustomerID"):
“Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 196 of `x` matches multiple rows in `y`.
ℹ Row 1 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship =
  "many-to-many"` to silence this warning.”



================ FINAL DATASET DIMENSIONS ================
Rows: 393617 
Columns: 9 

Column names:
[1] "InvoiceNo"   "StockCode"   "CustomerID"  "Quantity"    "InvoiceDate"
[6] "Description" "UnitPrice"   "Country"     "Revenue"    

================ UNMATCHED PRODUCTS ================
Number of unmatched product records: 0 

================ UNMATCHED CUSTOMERS ================
Number of unmatched customer records: 0 

================ INTEGRATED DATA CHECK ================
  InvoiceNo   StockCode  CustomerID    Quantity InvoiceDate Description 
          0           0           0           0           0           0 
  UnitPrice     Country     Revenue 
          0           0           0 

================ FINAL INTEGRATED DATA ================
# A tibble: 10 × 9
   InvoiceNo StockCode CustomerID Quantity InvoiceDate         Description      
   <chr>     <chr>          <dbl>    <dbl> <dttm>              <chr>            
 1 536365    85123A         17850        6 2010-12-01 08:26:

In [4]:
# Fix duplicate CustomerID mappings in customer data
customers_clean <- customers_clean %>%
  group_by(CustomerID) %>%
  summarise(
    Country = first(Country),
    .groups = "drop"
  )

# Recreate the integrated dataset
final_retail_data <- transactions_clean %>%
  left_join(
    products_clean,
    by = "StockCode",
    suffix = c("_transaction", "_product")
  ) %>%
  left_join(
    customers_clean,
    by = "CustomerID"
  ) %>%
  select(
    InvoiceNo,
    StockCode,
    CustomerID,
    Quantity,
    InvoiceDate,
    Description,
    UnitPrice = UnitPrice_transaction,
    Country,
    Revenue
  )

cat("Corrected integrated dataset:\n")
cat("Rows:", nrow(final_retail_data), "\n")
cat("Columns:", ncol(final_retail_data), "\n")

cat("\nMissing values:\n")
print(colSums(is.na(final_retail_data)))

Corrected integrated dataset:
Rows: 392708 
Columns: 9 

Missing values:
  InvoiceNo   StockCode  CustomerID    Quantity InvoiceDate Description 
          0           0           0           0           0           0 
  UnitPrice     Country     Revenue 
          0           0           0 


In [5]:
# ============================================================
# TASK 3: SALES AND CUSTOMER ANALYSIS
# ============================================================

# ------------------------------------------------------------
# 1. TOTAL SALES REVENUE
# ------------------------------------------------------------

total_revenue <- final_retail_data %>%
  summarise(
    Total_Revenue = sum(Revenue, na.rm = TRUE)
  )

cat("\n================ TOTAL SALES REVENUE ================\n")
print(total_revenue)


# ------------------------------------------------------------
# 2. TOP 5 PRODUCTS BASED ON REVENUE
# ------------------------------------------------------------

top_5_products <- final_retail_data %>%
  group_by(StockCode, Description) %>%
  summarise(
    Total_Revenue = sum(Revenue, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(desc(Total_Revenue)) %>%
  slice_head(n = 5)

cat("\n================ TOP 5 PRODUCTS ================\n")
print(top_5_products)


# ------------------------------------------------------------
# 3. TOP 5 COUNTRIES BASED ON REVENUE
# ------------------------------------------------------------

top_5_countries <- final_retail_data %>%
  group_by(Country) %>%
  summarise(
    Total_Revenue = sum(Revenue, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(desc(Total_Revenue)) %>%
  slice_head(n = 5)

cat("\n================ TOP 5 COUNTRIES ================\n")
print(top_5_countries)


# ------------------------------------------------------------
# 4. TOP 5 CUSTOMERS BASED ON TOTAL PURCHASE VALUE
# ------------------------------------------------------------

top_5_customers <- final_retail_data %>%
  group_by(CustomerID) %>%
  summarise(
    Total_Purchase_Value = sum(Revenue, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(desc(Total_Purchase_Value)) %>%
  slice_head(n = 5)

cat("\n================ TOP 5 CUSTOMERS ================\n")
print(top_5_customers)


# ------------------------------------------------------------
# 5. CUSTOMER VALUE CLASSIFICATION USING case_when()
# ------------------------------------------------------------

customer_value <- final_retail_data %>%
  group_by(CustomerID) %>%
  summarise(
    Total_Purchase_Value = sum(Revenue, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  mutate(
    Customer_Category = case_when(
      Total_Purchase_Value < 500 ~ "Low Value",
      Total_Purchase_Value < 2000 ~ "Medium Value",
      Total_Purchase_Value < 10000 ~ "High Value",
      TRUE ~ "Premium"
    )
  )


cat("\n================ CUSTOMER VALUE CLASSIFICATION ================\n")
print(head(customer_value, 10))


# ------------------------------------------------------------
# 6. NUMBER OF CUSTOMERS IN EACH CATEGORY
# ------------------------------------------------------------

customer_category_summary <- customer_value %>%
  count(Customer_Category) %>%
  arrange(desc(n))

cat("\n================ CUSTOMER CATEGORY SUMMARY ================\n")
print(customer_category_summary)


# ------------------------------------------------------------
# 7. REVENUE BY MARKET
# ------------------------------------------------------------

market_performance <- final_retail_data %>%
  group_by(Country) %>%
  summarise(
    Total_Revenue = sum(Revenue, na.rm = TRUE),
    Total_Transactions = n(),
    .groups = "drop"
  ) %>%
  arrange(desc(Total_Revenue))


# ------------------------------------------------------------
# 8. HIGH-PERFORMING MARKET
# ------------------------------------------------------------

high_performing_market <- market_performance %>%
  slice_max(
    order_by = Total_Revenue,
    n = 1
  )

cat("\n================ HIGH-PERFORMING MARKET ================\n")
print(high_performing_market)


# ------------------------------------------------------------
# 9. UNDERPERFORMING MARKET
# ------------------------------------------------------------

underperforming_market <- market_performance %>%
  slice_min(
    order_by = Total_Revenue,
    n = 1
  )

cat("\n================ UNDERPERFORMING MARKET ================\n")
print(underperforming_market)


# ------------------------------------------------------------
# 10. INTERPRETATION
# ------------------------------------------------------------

cat("\n================ MARKET INTERPRETATION ================\n")

cat(
  "\nThe high-performing market is",
  high_performing_market$Country,
  "with revenue of",
  round(high_performing_market$Total_Revenue, 2),
  ".\n"
)

cat(
  "It can be considered high-performing because it generated the highest\n",
  "revenue among all countries in the dataset.\n"
)

cat(
  "\nThe underperforming market is",
  underperforming_market$Country,
  "with revenue of",
  round(underperforming_market$Total_Revenue, 2),
  ".\n"
)

cat(
  "It is considered underperforming because it generated the lowest\n",
  "revenue among the countries represented in the dataset.\n"
)


# ------------------------------------------------------------
# 11. DISPLAY COMPLETE CUSTOMER ANALYSIS
# ------------------------------------------------------------

cat("\n================ CUSTOMER ANALYSIS COMPLETE ================\n")

print(customer_value)


================ TOTAL SALES REVENUE ================
# A tibble: 1 × 1
  Total_Revenue
          <dbl>
1     10174930.

================ TOP 5 PRODUCTS ================
# A tibble: 5 × 3
  StockCode Description                        Total_Revenue
  <chr>     <chr>                                      <dbl>
1 22502     PICNIC BASKET WICKER 60 PIECES           486672.
2 23843     PAPER CRAFT , LITTLE BIRDIE              168470.
3 22423     REGENCY CAKESTAND 3 TIER                 157896 
4 85123A    CREAM HANGING HEART T-LIGHT HOLDER       108451.
5 23166     MEDIUM CERAMIC TOP STORAGE JAR            97395 

================ TOP 5 COUNTRIES ================
# A tibble: 5 × 2
  Country        Total_Revenue
  <chr>                  <dbl>
1 United Kingdom      8429657.
2 Netherlands          335171.
3 EIRE                 327189.
4 Germany              238815.
5 France               208477.

================ TOP 5 CUSTOMERS ================
# A tibble: 5 × 2
  CustomerID Total_Purchase_V

In [6]:
# ============================================================
# TASK 4: SQLITE DATABASE AND SQL QUERIES
# ============================================================

# ------------------------------------------------------------
# STEP 1: CREATE SQLITE DATABASE
# ------------------------------------------------------------

db <- dbConnect(
  SQLite(),
  "retail_sales.db"
)


# ------------------------------------------------------------
# STEP 2: EXPORT FINAL DATASET TO retail_sales TABLE
# ------------------------------------------------------------

dbWriteTable(
  db,
  "retail_sales",
  final_retail_data,
  overwrite = TRUE
)

cat("\n================ DATABASE CREATED ================\n")
cat("Database: retail_sales.db\n")
cat("Table: retail_sales\n")


# ------------------------------------------------------------
# STEP 3: VERIFY TABLE
# ------------------------------------------------------------

cat("\n================ DATABASE TABLES ================\n")

print(dbListTables(db))


# ------------------------------------------------------------
# STEP 4: CHECK TABLE STRUCTURE
# ------------------------------------------------------------

cat("\n================ TABLE STRUCTURE ================\n")

print(dbGetQuery(
  db,
  "PRAGMA table_info(retail_sales)"
))


# ------------------------------------------------------------
# SQL QUERY 1: TOP 5 CUSTOMERS
# ------------------------------------------------------------

query1 <- "
SELECT
    CustomerID,
    ROUND(SUM(Revenue), 2) AS Total_Revenue
FROM retail_sales
GROUP BY CustomerID
ORDER BY Total_Revenue DESC
LIMIT 5
"

top_customers_sql <- dbGetQuery(db, query1)

cat("\n================ SQL QUERY 1 ================\n")
cat("Top 5 customers based on revenue:\n")
print(top_customers_sql)


# ------------------------------------------------------------
# SQL QUERY 2: TOTAL REVENUE BY COUNTRY
# ------------------------------------------------------------

query2 <- "
SELECT
    Country,
    ROUND(SUM(Revenue), 2) AS Total_Revenue
FROM retail_sales
GROUP BY Country
ORDER BY Total_Revenue DESC
"

country_revenue_sql <- dbGetQuery(db, query2)

cat("\n================ SQL QUERY 2 ================\n")
cat("Total revenue by country:\n")
print(head(country_revenue_sql, 10))


# ------------------------------------------------------------
# STEP 5: THREE BUSINESS INSIGHTS
# ------------------------------------------------------------

cat("\n================ BUSINESS INSIGHTS ================\n")

top_country <- country_revenue_sql %>%
  slice_max(Total_Revenue, n = 1)

top_customer <- top_customers_sql %>%
  slice_max(Total_Revenue, n = 1)

top_product <- top_5_products %>%
  slice_max(Total_Revenue, n = 1)


cat(
  "\nInsight 1 - Market:\n",
  top_country$Country,
  "is the highest revenue-generating country with revenue of",
  round(top_country$Total_Revenue, 2),
  ".\n"
)

cat(
  "\nInsight 2 - Customer:\n",
  "Customer",
  top_customer$CustomerID,
  "has the highest purchase value among customers in the dataset,\n",
  "with total revenue of",
  round(top_customer$Total_Revenue, 2),
  ".\n"
)

cat(
  "\nInsight 3 - Product:\n",
  top_product$Description,
  "is the highest revenue-generating product in the analysis,\n",
  "with revenue of",
  round(top_product$Total_Revenue, 2),
  ".\n"
)


# ------------------------------------------------------------
# STEP 6: VERIFY DATA STORED IN SQLITE
# ------------------------------------------------------------

cat("\n================ SQLITE RECORD COUNT ================\n")

print(
  dbGetQuery(
    db,
    "SELECT COUNT(*) AS Total_Rows FROM retail_sales"
  )
)


# ------------------------------------------------------------
# STEP 7: CLOSE DATABASE CONNECTION
# ------------------------------------------------------------

dbDisconnect(db)

cat("\nSQLite database connection closed successfully.\n")
cat("retail_sales.db has been created successfully.\n")


================ DATABASE CREATED ================
Database: retail_sales.db
Table: retail_sales

================ DATABASE TABLES ================
[1] "retail_sales"

================ TABLE STRUCTURE ================
  cid        name type notnull dflt_value pk
1   0   InvoiceNo TEXT       0         NA  0
2   1   StockCode TEXT       0         NA  0
3   2  CustomerID REAL       0         NA  0
4   3    Quantity REAL       0         NA  0
5   4 InvoiceDate REAL       0         NA  0
6   5 Description TEXT       0         NA  0
7   6   UnitPrice REAL       0         NA  0
8   7     Country TEXT       0         NA  0
9   8     Revenue REAL       0         NA  0

================ SQL QUERY 1 ================
Top 5 customers based on revenue:
  CustomerID Total_Revenue
1      18102      403024.2
2      14646      329710.5
3      17450      181056.5
4      14156      170378.0
5      16446      168472.5

================ SQL QUERY 2 ================
Total revenue by country:
          Count